# Weeks 8–9 on Colab: Prompt Guard baselines + fine-tuned classifier

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Total time is roughly 30–45 minutes. The data splits are already in the repo, so there's no need to rebuild the dataset here.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Get the code
Replace the URL with your GitHub repo. For a private repo, use a GitHub personal access token in the URL or make the repo public temporarily.

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/pi-detector-thesis.git"  # <- change this
!git clone -q $REPO_URL
%cd pi-detector-thesis
!git log --oneline | head -3

In [ ]:
# Colab already has torch with GPU support; this adds the rest
!pip install -q 'transformers>=4.44' 'accelerate>=0.30' sentencepiece scikit-learn

## 2. Log in to Hugging Face (for Prompt Guard)
Prompt Guard is gated. Once: open https://huggingface.co/meta-llama/Llama-Prompt-Guard-2-86M and https://huggingface.co/meta-llama/Prompt-Guard-86M, accept the license, then create a **Read** token at https://huggingface.co/settings/tokens. Paste the token when asked below. Skip this cell to run only the ProtectAI detector.

In [ ]:
from huggingface_hub import login
login()

## 3. Week 8: local baselines (keyword filter, TF-IDF), about 1 minute

In [ ]:
!python scripts/eval_baselines.py | tail -5

## 4. Week 8: off-the-shelf detectors (Prompt Guard 1 & 2, ProtectAI), about 5–10 minutes

In [ ]:
!python scripts/eval_hf_detectors.py

## 5. Week 9: fine-tune DeBERTa-v3-small with chunking, about 15–25 minutes

In [ ]:
!python scripts/train_classifier.py

## 6. Ablation: same model without chunking (whole doc, truncated at 512 tokens)
This tells you whether chunking actually helps. Optional if you're short on time.

In [ ]:
!python scripts/train_classifier.py --no-chunking | tail -3

## 7. Look at the results and save them back to GitHub

In [ ]:
from IPython.display import Markdown
Markdown(open('results/week9_comparison.md').read())

In [ ]:
# Save results (not the model weights) to your repo.
# You need a GitHub token with write access for the push to work from Colab.
!git add results/
!git -c user.name='Gary' -c user.email='you@example.com' commit -m 'Week 8-9 results from Colab' && git push

### Keep the trained model (optional)
Colab deletes files when the session ends. To keep the model for week 10–11, copy it to Google Drive:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/thesis && cp -r models/deberta-v3-small /content/drive/MyDrive/thesis/